# Synthetic repeated-facies SC-PaCMAP benchmark

## Study overview

**Research question.** Under the same experiment used by the CMiO benchmark,
does SC-PaCMAP produce K-means partitions with a better balance of spatial
contiguity and multivariate delineation than coordinate-aware UMAP, PaCMAP,
LocalMAP, and SpaSNE?

The synthetic answer is deliberately easy to understand. A slightly jittered
$32\times32$ grid is divided into sixteen $8\times8$ patches. Four facies obey
the Latin-square rule

$$
\text{facies}=(\text{tile}_x+\text{tile}_y)\bmod4,
$$

so every facies appears in four disconnected patches.

The dataset imposes two traps. First, only 10 of 50 variables contain the
facies signal; 20 mutually dependent nonlinear variables form a strong but
spatially shuffled nuisance manifold, and another 20 variables are independent
noise. Second, geographic proximity helps within each patch but cannot by
itself identify that distant patches share a facies. A literal map-preserving
solution may retain sixteen locations instead of recovering four repeated
regimes.

Facies labels are excluded from preprocessing, embedding, clustering, and
parameter selection. They are used only for visualization and final adjusted
Rand index (ARI) evaluation. The notebook first isolates the PaCMAP design
choices in a focused ablation, then evaluates all six candidates using the same
CMiO clustering and spatial-evaluation workflow. This is a designed mechanism
stress test, not evidence of universal superiority.

In [ ]:
from __future__ import annotations

import hashlib
import os
import platform
import sys
import time
from importlib.metadata import version
from pathlib import Path

os.environ["NUMBA_NUM_THREADS"] = "1"

import matplotlib.pyplot as plt
import numba
import numpy as np
import pacmap
import pandas as pd
import seaborn as sns
import umap
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


def find_repository_root(start: Path) -> Path:
    ancestors = (start.resolve(), *start.resolve().parents)
    candidates = (*ancestors, *(path / "sc-pacmap-reproducibility" for path in ancestors))
    for candidate in candidates:
        if (
            (candidate / "code" / "sc_pacmap" / "sc_pacmap.py").is_file()
            and (candidate / "data" / "processed").is_dir()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the sc-pacmap-reproducibility repository.")


ROOT = find_repository_root(Path.cwd())
SC_MODULE_DIR = ROOT / "code" / "sc_pacmap"
SPASNE_MODULE_DIR = ROOT / "code" / "spasne"
for module_dir in (SC_MODULE_DIR, SPASNE_MODULE_DIR):
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

from spasne_adapter import python_spasne
from sc_pacmap import (
    PaCMAPSettings,
    SCPaCMAPConfig,
    run_legacy_sc_pacmap,
    run_sc_pacmap,
    verify_library_matches_copied_code,
    verify_sc_disabled_matches_original,
)

if version("pacmap") != "0.9.1":
    raise RuntimeError("This benchmark requires pacmap==0.9.1.")
if numba.config.NUMBA_NUM_THREADS != 1:
    raise RuntimeError("Restart the kernel so NUMBA_NUM_THREADS=1 applies before import.")
numba.set_num_threads(1)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

print(f"Python: {platform.python_version()}")
print(f"PaCMAP: {version('pacmap')}; UMAP: {version('umap-learn')}")
print(f"Repository: {ROOT}")

## Unified experiment configuration

This notebook uses the same deterministic CMiO workflow structure, with a
shared compact 8-neighbour graph for every local-neighbour method: seed 42,
two embedding dimensions, Euclidean distance,
PaCMAP-family `MN_ratio=0.5`, `FP_ratio=2.0`, learning rate 1.0, and
optimization phases of $(100,100,250)$. The known synthetic answer contains
four facies, while the complete clustering sweep remains $K=2,\ldots,10$.

In [ ]:
# Unified deterministic benchmark parameters, matched to the CMiO contract
SEED = 42
GRID_SIDE = 32
TILES_PER_SIDE = 4
TILE_SIDE = GRID_SIDE // TILES_PER_SIDE
N_SAMPLES = GRID_SIDE**2
N_FEATURES = 50
N_NEIGHBORS = 8
N_COMPONENTS = 2
KNOWN_CLASS_COUNT = 4
K_VALUES = range(2, 11)

SPASNE_ALPHA = 10.0
SPASNE_BETA = 5.0
SPASNE_EXAGGERATION = 4
SPASNE_PERPLEXITY = 50
SPASNE_THETA = 0.5
SPASNE_MAX_ITER = 1000
SPASNE_DIR = ROOT / "code" / "spasne" / "spasne_original"

XY_COLUMNS = ["x_m", "y_m"]
LABEL_COLUMN = "truth_facies"
SPATIAL_FEATURES = [f"spatial_{index:02d}" for index in range(1, 11)]
CONFOUNDER_FEATURES = [f"confounder_{index:02d}" for index in range(1, 21)]
NOISE_FEATURES = [f"noise_{index:02d}" for index in range(1, 21)]
FEATURE_COLUMNS = SPATIAL_FEATURES + CONFOUNDER_FEATURES + NOISE_FEATURES
METADATA_COLUMNS = [
    "sample_id",
    *XY_COLUMNS,
    LABEL_COLUMN,
    "facies_id",
    "patch_id",
]
EXPECTED_LABELS = {
    0: "Facies A",
    1: "Facies B",
    2: "Facies C",
    3: "Facies D",
}

spasne_paths = {
    "upstream Python wrapper": SPASNE_DIR / "spasne.py",
    "project WSL bridge": SPASNE_DIR.parent / "spasne_wsl_bridge.py",
    "locally built Linux executable": SPASNE_DIR / "spasne",
}
missing_spasne_paths = [str(path) for path in spasne_paths.values() if not path.is_file()]
if missing_spasne_paths:
    raise RuntimeError(
        "SpaSNE is not built. Run 'make' in code/spasne/spasne_original "
        f"from Linux or WSL. Missing: {missing_spasne_paths}"
    )
assert len(FEATURE_COLUMNS) == N_FEATURES

spasne_configuration = pd.Series(
    {
        "alpha": SPASNE_ALPHA,
        "beta": SPASNE_BETA,
        "exaggeration": SPASNE_EXAGGERATION,
        "perplexity": SPASNE_PERPLEXITY,
        "theta": SPASNE_THETA,
        "max_iter": SPASNE_MAX_ITER,
        "use_pca": True,
        "random_state": SEED,
    },
    name="SpaSNE published-default configuration",
)
display(spasne_configuration.to_frame())

print(
    f"CMiO-matched setup: seed={SEED}, neighbors={N_NEIGHBORS}, "
    f"dimensions={N_COMPONENTS}, K={min(K_VALUES)}-{max(K_VALUES)}."
)

## Data inspection

The generator creates 1,024 samples, raw metre-based XY coordinates, and 50
numeric variables. The four-facies answer, the competing feature structures,
and the role of each block are fixed before any embedding is fitted.

| Block | Columns | Construction | Intended challenge |
|---|---:|---|---|
| Spatial signal | 1–10 | Balanced facies contrasts with Gaussian noise $\sigma=0.15,\ldots,0.95$ | Ten genuine spatial signals with different autocorrelation intensity |
| Nonlinear confounders | 11–30 | Periodic, polynomial, interaction, and saturating transforms of sample-wise $\theta$ and $q$ | Strong dependent feature similarity with no spatial meaning |
| Independent distractors | 31–50 | IID standard Gaussian | Twenty irrelevant dimensions dilute the minority signal |

The data contract below verifies the complete population, schema, balanced
answer, and finite numeric values. No truth or patch column enters the
50-feature matrix.

In [ ]:
def make_synthetic_repeated_facies(
    random_state: int = SEED,
) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    """Create the repeated-facies benchmark and its plotting-only latent values."""
    # 1. Use one local random generator so the complete dataset is reproducible.
    rng = np.random.default_rng(random_state)

    # 2. Create a 32 x 32 sampling grid, then add about 1 m of coordinate jitter.
    #    Adjacent nominal samples remain 25 m apart, but the layout is not perfectly regular.
    grid_axis = np.arange(GRID_SIDE)
    x_grid, y_grid = np.meshgrid(grid_axis, grid_axis)
    x_index = x_grid.ravel().astype(np.int64)
    y_index = y_grid.ravel().astype(np.int64)
    coordinate_indices = np.column_stack([x_index, y_index]).astype(np.float64)
    coordinate_jitter = rng.normal(0.0, 0.04, size=coordinate_indices.shape)
    coordinates = 25.0 * (coordinate_indices + coordinate_jitter)

    # 3. Divide the grid into sixteen 8 x 8 physical patches.
    tile_x = x_index // TILE_SIDE
    tile_y = y_index // TILE_SIDE
    patch_id = tile_y * TILES_PER_SIDE + tile_x

    # Apply a Latin-square rule. Each facies occurs in four disconnected patches.
    facies_id = (tile_x + tile_y) % KNOWN_CLASS_COUNT

    # 4. Give every facies a balanced three-value signature.
    #    Any two facies differ in exactly two of the three signature values.
    facies_prototype = np.array(
        [
            [-1.0, -1.0, -1.0],  # Facies A
            [-1.0, +1.0, +1.0],  # Facies B
            [+1.0, -1.0, +1.0],  # Facies C
            [+1.0, +1.0, -1.0],  # Facies D
        ],
        dtype=np.float64,
    )

    # Build ten answer-bearing features from the three-value signatures.
    # Features cycle through signature positions 0, 1, 2 while their noise increases.
    spatial_noise_sigma = np.linspace(0.15, 0.95, 10)
    spatial_columns = []
    for feature_index, noise_sigma in enumerate(spatial_noise_sigma):
        prototype_position = feature_index % facies_prototype.shape[1]
        facies_signal = facies_prototype[facies_id, prototype_position]
        measurement_noise = noise_sigma * rng.normal(size=N_SAMPLES)
        spatial_columns.append(facies_signal + measurement_noise)
    spatial_block = np.column_stack(spatial_columns)

    # 5. Draw nuisance values independently for each sample. They have no intended
    #    relationship with facies or location, but their nonlinear structure is strong.
    nuisance_theta = rng.uniform(-np.pi, np.pi, size=N_SAMPLES)
    nuisance_q = rng.normal(size=N_SAMPLES)
    nuisance_radius = 1.0 + 0.18 * rng.normal(size=N_SAMPLES)

    # Name repeated transformations so the twenty confounders are easier to read.
    cos_theta = np.cos(nuisance_theta)
    sin_theta = np.sin(nuisance_theta)
    tanh_q = np.tanh(nuisance_q)
    centered_q_squared = nuisance_q**2 - 1.0

    confounder_basis = np.column_stack(
        [
            # A noisy circular manifold and its second and third harmonics.
            nuisance_radius * cos_theta,
            nuisance_radius * sin_theta,
            np.cos(2.0 * nuisance_theta),
            np.sin(2.0 * nuisance_theta),
            np.cos(3.0 * nuisance_theta),
            np.sin(3.0 * nuisance_theta),
            # Saturating, polynomial, and periodic transforms of q.
            tanh_q,
            centered_q_squared,
            np.sin(1.5 * nuisance_q),
            np.cos(1.5 * nuisance_q),
            # Additive mixtures of the angle and q structures.
            cos_theta + 0.55 * tanh_q,
            sin_theta - 0.55 * tanh_q,
            np.cos(2.0 * nuisance_theta) + 0.35 * centered_q_squared,
            np.sin(2.0 * nuisance_theta) - 0.35 * centered_q_squared,
            # Multiplicative interactions between the angle and q structures.
            tanh_q * cos_theta,
            tanh_q * sin_theta,
            centered_q_squared * cos_theta,
            centered_q_squared * sin_theta,
            # Two final nonlinear combinations.
            np.sin(nuisance_theta + nuisance_q),
            np.cos(nuisance_theta - nuisance_q),
        ]
    )
    confounder_noise = rng.normal(0.0, 0.12, size=confounder_basis.shape)
    confounder_block = confounder_basis + confounder_noise

    # 6. Add twenty completely independent distractor features.
    independent_block = rng.normal(size=(N_SAMPLES, 20))

    # 7. Join the three feature blocks in their documented 10 + 20 + 20 order.
    feature_matrix = np.column_stack(
        [spatial_block, confounder_block, independent_block]
    )
    assert feature_matrix.shape == (N_SAMPLES, N_FEATURES)

    # Keep labels and locations as metadata; they are not part of FEATURE_COLUMNS.
    metadata = pd.DataFrame(
        {
            "sample_id": [f"S{index:04d}" for index in range(N_SAMPLES)],
            "x_m": coordinates[:, 0],
            "y_m": coordinates[:, 1],
            LABEL_COLUMN: [EXPECTED_LABELS[value] for value in facies_id],
            "facies_id": facies_id,
            "patch_id": patch_id,
        }
    )
    feature_data = pd.DataFrame(feature_matrix, columns=FEATURE_COLUMNS)
    generated = pd.concat([metadata, feature_data], axis=1)

    # Return latent values only for diagnostics and plotting.
    latent = {
        "nuisance_theta": nuisance_theta,
        "spatial_noise_sigma": spatial_noise_sigma,
    }
    return generated, latent

# 8. Generate the complete synthetic population once.
data, latent = make_synthetic_repeated_facies()

# 9. Verify the row count, schema, identifiers, and required values.
assert data.shape == (N_SAMPLES, len(METADATA_COLUMNS) + N_FEATURES)
assert data.columns.tolist() == METADATA_COLUMNS + FEATURE_COLUMNS
assert not data["sample_id"].duplicated().any()
assert not data[METADATA_COLUMNS + FEATURE_COLUMNS].isna().any().any()

# Separate the benchmark inputs from evaluation-only labels.
features_raw = data[FEATURE_COLUMNS].to_numpy(dtype=np.float64)
coordinates_raw = data[XY_COLUMNS].to_numpy(dtype=np.float64)
labels_true = data["facies_id"].to_numpy(dtype=np.int64)
label_names = data[LABEL_COLUMN].astype(str).to_numpy()

# Check numeric validity and confirm all four expected facies are present.
assert np.isfinite(features_raw).all() and np.isfinite(coordinates_raw).all()
assert np.all(np.ptp(features_raw, axis=0) > 0)
assert set(np.unique(labels_true)) == set(EXPECTED_LABELS)
observed_labels = (
    data[["facies_id", LABEL_COLUMN]]
    .drop_duplicates()
    .set_index("facies_id")[LABEL_COLUMN]
    .to_dict()
)
assert observed_labels == EXPECTED_LABELS

# 10. Standardize features and coordinates using separate scalers.
# Match the CMiO benchmark's feature precision and TSS calculation.
feature_scaler = StandardScaler()
coordinate_scaler = StandardScaler()
features_standardized = feature_scaler.fit_transform(features_raw).astype(np.float32)
coordinates_standardized = coordinate_scaler.fit_transform(coordinates_raw)
features_with_xy = np.column_stack(
    [features_standardized, coordinates_standardized]
)

# TSS is the total feature variation used later to normalize clustering WCSS.
feature_center = features_standardized.mean(axis=0, keepdims=True)
total_sum_squares = float(
    np.sum((features_standardized - feature_center) ** 2)
)
assert total_sum_squares > 0

# 11. Display compact checks that make the generated-data contract visible.
contract = pd.Series(
    {
        "rows": len(data),
        "numeric features": len(FEATURE_COLUMNS),
        "coordinate columns": len(XY_COLUMNS),
        "known facies": data[LABEL_COLUMN].nunique(),
        "physical patches": data["patch_id"].nunique(),
        "duplicate sample IDs": int(data["sample_id"].duplicated().sum()),
        "missing required values": int(
            data[METADATA_COLUMNS + FEATURE_COLUMNS].isna().sum().sum()
        ),
    },
    name="synthetic repeated-facies data contract",
)
display(contract.to_frame())
display(
    data.groupby(["facies_id", LABEL_COLUMN], observed=True)
    .size()
    .rename("samples")
    .reset_index()
)
display(
    pd.DataFrame(
        {
            "feature block": [
                "spatial signal",
                "nonlinear dependent confounder",
                "independent distractor",
            ],
            "columns": [10, 20, 20],
            "answer-bearing": [True, False, False],
            "spatially generated": [True, False, False],
        }
    )
)

### Spatial support and held-out class balance

The first panel reveals the held-out four-facies answer. The second shows a
representative answer-bearing spatial variable; its progressively noisier
companions create different spatial-correlation intensities. The third shows
the spatially shuffled nonlinear nuisance angle. The final panel confirms
equal class size. All four views are descriptive only: neither facies nor patch
identity is supplied to any embedding or clustering step.

In [ ]:
class_ids = list(EXPECTED_LABELS)
class_colors = dict(
    zip(class_ids, sns.color_palette("colorblind", len(class_ids)))
)

fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
for class_id in class_ids:
    mask = labels_true == class_id
    axes[0, 0].scatter(
        coordinates_raw[mask, 0],
        coordinates_raw[mask, 1],
        s=13,
        alpha=0.82,
        color=class_colors[class_id],
        label=EXPECTED_LABELS[class_id],
        linewidth=0,
    )
for boundary in (7.5, 15.5, 23.5):
    axes[0, 0].axvline(25.0 * boundary, color="black", linewidth=0.6, alpha=0.35)
    axes[0, 0].axhline(25.0 * boundary, color="black", linewidth=0.6, alpha=0.35)
axes[0, 0].set_title("Known four-facies answer")
axes[0, 0].legend(frameon=False, markerscale=1.4)

signal_plot = axes[0, 1].scatter(
    coordinates_raw[:, 0],
    coordinates_raw[:, 1],
    c=features_standardized[:, 0],
    cmap="viridis",
    s=11,
    linewidth=0,
)
axes[0, 1].set_title(
    f"Representative spatial signal\n"
    f"{SPATIAL_FEATURES[0]}, sigma={latent['spatial_noise_sigma'][0]:.2f}"
)
fig.colorbar(signal_plot, ax=axes[0, 1], label="standardized value")

nuisance_plot = axes[1, 0].scatter(
    coordinates_raw[:, 0],
    coordinates_raw[:, 1],
    c=latent["nuisance_theta"],
    cmap="twilight",
    s=11,
    linewidth=0,
)
axes[1, 0].set_title("Non-spatial nonlinear nuisance")
fig.colorbar(nuisance_plot, ax=axes[1, 0], label="latent angle theta")

counts = data["facies_id"].value_counts().reindex(class_ids)
axes[1, 1].barh(
    [EXPECTED_LABELS[class_id] for class_id in class_ids],
    counts,
    color=[class_colors[class_id] for class_id in class_ids],
)
axes[1, 1].set(title="Class counts", xlabel="Samples", ylabel="")
axes[1, 1].invert_yaxis()

for axis in axes.flat[:3]:
    axis.set_xlabel("X (m)")
    axis.set_ylabel("Y (m)")
    axis.set_aspect("equal")
fig.suptitle("Synthetic answer, spatial signal, and nuisance trap", fontsize=15)
plt.show()

## Candidate embeddings

All candidates use two output dimensions and seed 42. UMAP and the PaCMAP
family use the same compact 8-neighbour graph and Euclidean distance;
PaCMAP-family candidates also
use learning rate 1.0, `MN_ratio=0.5`, `FP_ratio=2.0`, and optimization phases
of $(100,100,250)$. SpaSNE uses its published defaults without label-guided
tuning: $\alpha=10$, $\beta=5$, perplexity 50, exaggeration 4, $\theta=0.5$,
PCA preprocessing, and 1000 iterations.

- **PaCMAP feature-only** uses the 50 standardized features and no coordinates.
- **PaCMAP + XY** appends separately standardized XY coordinates to the feature matrix.
- **SC-PaCMAP (neighbour replacement graph)** replaces part of the PaCMAP neighbour graph with spatially selected PCA-Geary edges.
- **SC-PaCMAP (full graph)** retains the complete feature-neighbour graph, constructs a separate PCA-Geary spatial graph from raw XY, and ramps its attraction only in Phase 3.
- **SpaSNE** receives standardized features and raw XY separately through its feature, global-feature, and spatial terms.
- **UMAP + XY** uses standardized features plus standardized XY.
- **LocalMAP + XY** uses standardized features plus standardized XY.

The intended answer is four repeated facies, not sixteen geographic patches.
Feature-only methods can follow the nonlinear nuisance; unconditional spatial
preservation can retain the mosaic. The two SC-PaCMAP structures test whether a small,
feature-compatible spatial graph improves the final partition while the full
feature graph remains available to connect repeated facies.

The first results section is the focused PaCMAP ablation: feature-only PaCMAP,
PaCMAP + XY, and both SC-PaCMAP variants. The second section retains all seven candidates.

### PaCMAP implementation check

Before fitting the candidates, SC-PaCMAP with its spatial force disabled must
reproduce the copied PaCMAP implementation exactly. The copied implementation
is also compared with the installed `pacmap==0.9.1` library.

In [ ]:
settings = PaCMAPSettings(
    n_components=N_COMPONENTS,
    n_neighbors=N_NEIGHBORS,
    mn_ratio=0.5,
    fp_ratio=2.0,
    learning_rate=1.0,
    num_iters=(100, 100, 250),
    random_state=SEED,
    knn_backend="faiss",
    verbose=False,
)
sc_config = SCPaCMAPConfig()
audit_index = np.linspace(0, len(data) - 1, 200, dtype=int)
audit_features = np.ascontiguousarray(features_standardized[audit_index])
audit_coordinates = np.ascontiguousarray(coordinates_raw[audit_index])

sc_disabled_audit = verify_sc_disabled_matches_original(
    audit_features, audit_coordinates, settings
)
library_audit = verify_library_matches_copied_code(audit_features, settings)

required_equalities = [
    "neighbor_pairs_equal",
    "mid_near_pairs_equal",
    "further_pairs_equal",
    "embedding_equal",
]
assert all(sc_disabled_audit[key] for key in required_equalities)
assert sc_disabled_audit["maximum_embedding_error"] == 0.0
assert library_audit["status"] == "pass"
assert all(library_audit[key] for key in required_equalities)
assert library_audit["maximum_embedding_error"] == 0.0

audit_table = pd.DataFrame(
    [
        {"audit": "SC disabled vs copied PaCMAP", **sc_disabled_audit},
        {"audit": "copied PaCMAP vs installed library", **library_audit},
    ]
)
display(audit_table)

### Fitted embeddings

Each candidate is fitted to the full synthetic population. The output table
records input pathways and runtime; the following diagnostic view colors the
embeddings by the held-out facies for inspection only.

In [ ]:
METHOD_ORDER = [
    "UMAP + XY",
    "PaCMAP + XY",
    "LocalMAP + XY",
    "PaCMAP feature-only",
    "SC-PaCMAP (neighbour replacement graph)",
    "SC-PaCMAP (full graph)",
    "SpaSNE",
]
embeddings = {}
runtimes = {}


def store_embedding(method: str, embedding: np.ndarray, started_at: float) -> None:
    embedding = np.asarray(embedding, dtype=np.float64)
    assert embedding.shape == (len(data), N_COMPONENTS)
    assert np.isfinite(embedding).all()
    embeddings[method] = embedding
    runtimes[method] = time.perf_counter() - started_at


def fit_localmap_in_clean_process(
    x_input: np.ndarray,
    n_neighbors: int,
) -> np.ndarray:
    import subprocess

    worker_code = r'''
import os
os.environ["NUMBA_NUM_THREADS"] = "1"
import importlib
import sys
import numba
import numpy as np
import pacmap

@numba.njit
def seed_numba_rng(random_state):
    np.random.seed(random_state)

pacmap_core = importlib.import_module("pacmap.pacmap")
original_resample = pacmap_core.sample_FP_pair_nearby
resample_call_index = 0

def deterministic_resample(X, pair_neighbors, old_pair_FP, Y, low_dist_thres):
    global resample_call_index
    seed_numba_rng(42 + resample_call_index)
    resample_call_index += 1
    return original_resample(X, pair_neighbors, old_pair_FP, Y, low_dist_thres)

pacmap_core.sample_FP_pair_nearby = deterministic_resample
rows, columns, n_neighbors = map(int, sys.argv[1:4])
x_input = np.frombuffer(sys.stdin.buffer.read(), dtype=np.float64).reshape(rows, columns)
seed_numba_rng(42)
model = pacmap.LocalMAP(
    n_components=2, n_neighbors=n_neighbors, MN_ratio=0.5, FP_ratio=2.0,
    distance="euclidean", lr=1.0, num_iters=(100, 100, 250),
    apply_pca=True, random_state=42, save_tree=False, low_dist_thres=10,
    knn_backend="faiss",
)
embedding = model.fit_transform(x_input, init="pca", save_pairs=False)
sys.stdout.buffer.write(np.asarray(embedding, dtype=np.float32).tobytes())
'''
    x_input = np.ascontiguousarray(x_input, dtype=np.float64)
    completed = subprocess.run(
        [
            sys.executable,
            "-c",
            worker_code,
            *map(str, x_input.shape),
            str(n_neighbors),
        ],
        input=x_input.tobytes(),
        capture_output=True,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.decode("utf-8", errors="replace"))
    expected_bytes = len(x_input) * N_COMPONENTS * np.dtype(np.float32).itemsize
    if len(completed.stdout) != expected_bytes:
        raise RuntimeError("LocalMAP worker returned an unexpected embedding size.")
    return np.frombuffer(completed.stdout, dtype=np.float32).reshape(
        len(x_input), N_COMPONENTS
    ).copy()


started_at = time.perf_counter()
umap_model = umap.UMAP(
    n_neighbors=N_NEIGHBORS,
    n_components=N_COMPONENTS,
    metric="euclidean",
    learning_rate=1.0,
    random_state=SEED,
    n_jobs=1,
)
store_embedding(
    "UMAP + XY", umap_model.fit_transform(features_with_xy), started_at
)

pacmap_parameters = {
    "n_components": N_COMPONENTS,
    "n_neighbors": N_NEIGHBORS,
    "MN_ratio": 0.5,
    "FP_ratio": 2.0,
    "distance": "euclidean",
    "lr": 1.0,
    "num_iters": (100, 100, 250),
    "apply_pca": True,
    "random_state": SEED,
    "save_tree": False,
    "knn_backend": "faiss",
}

started_at = time.perf_counter()
pacmap_xy_model = pacmap.PaCMAP(**pacmap_parameters)
store_embedding(
    "PaCMAP + XY",
    pacmap_xy_model.fit_transform(
        features_with_xy, init="pca", save_pairs=False
    ),
    started_at,
)

started_at = time.perf_counter()
store_embedding(
    "LocalMAP + XY",
    fit_localmap_in_clean_process(features_with_xy, N_NEIGHBORS),
    started_at,
)

started_at = time.perf_counter()
pacmap_feature_model = pacmap.PaCMAP(**pacmap_parameters)
store_embedding(
    "PaCMAP feature-only",
    pacmap_feature_model.fit_transform(
        features_standardized, init="pca", save_pairs=False
    ),
    started_at,
)

started_at = time.perf_counter()
replacement_sc_result, replacement_sc_diagnostics = run_legacy_sc_pacmap(
    features_standardized,
    coordinates_raw,
    settings,
)
store_embedding(
    "SC-PaCMAP (neighbour replacement graph)", replacement_sc_result.embedding, started_at
)
assert replacement_sc_diagnostics is not None

started_at = time.perf_counter()
sc_result, sc_diagnostics = run_sc_pacmap(
    features_standardized,
    coordinates_raw,
    settings,
    config=sc_config,
)
store_embedding("SC-PaCMAP (full graph)", sc_result.embedding, started_at)
assert sc_diagnostics is not None
assert sc_diagnostics.k_attribute == N_NEIGHBORS
assert 0 <= sc_diagnostics.k_spatial <= N_NEIGHBORS
assert sc_diagnostics.pair_sc.shape == (
    len(data) * sc_diagnostics.k_spatial,
    2,
)
assert 0.0 <= sc_diagnostics.spatial_strength_lambda <= 1.0

started_at = time.perf_counter()
spasne_embedding = python_spasne(
    features_standardized,
    coordinates_raw,
    random_state=SEED,
    alpha=SPASNE_ALPHA,
    beta=SPASNE_BETA,
    exaggeration=SPASNE_EXAGGERATION,
    perplexity=SPASNE_PERPLEXITY,
    theta=SPASNE_THETA,
    max_iter=SPASNE_MAX_ITER,
    use_pca=True,
    spasne_dir=SPASNE_DIR,
)
store_embedding("SpaSNE", spasne_embedding, started_at)

embedding_cluster_inputs = embeddings
runtime_table = pd.DataFrame(
    {
        "method": METHOD_ORDER,
        "seconds": [runtimes[method] for method in METHOD_ORDER],
        "numeric feature columns": [N_FEATURES] * len(METHOD_ORDER),
        "coordinate pathway": [
            "standardized XY columns",
            "standardized XY columns",
            "standardized XY columns",
            "none",
            "raw XY replacement PCA-Geary graph",
            "raw XY separate Phase-3 graph and force",
            "raw XY SpaSNE spatial objective",
        ],
    }
)
display(runtime_table.round({"seconds": 3}))

### Embedding diagnostics and visual inspection

Embedding fingerprints, SC-PaCMAP diagnostics, and the held-out-facies
embedding view document the fitted candidates before evaluation.

In [ ]:
embedding_fingerprint_table = pd.DataFrame(
    {
        "method": METHOD_ORDER,
        "embedding SHA-256": [
            hashlib.sha256(
                np.ascontiguousarray(embeddings[method]).tobytes()
            ).hexdigest()
            for method in METHOD_ORDER
        ],
    }
)
display(embedding_fingerprint_table)

diagnostic_summary = pd.Series(
    {
        "spatial-strength lambda": sc_diagnostics.spatial_strength_lambda,
        "complete feature neighbors": sc_diagnostics.k_attribute,
        "separate SC edges per sample": sc_diagnostics.k_spatial,
        "Geary C minimum": np.min(sc_diagnostics.geary_c),
        "Geary C median": np.median(sc_diagnostics.geary_c),
        "Geary C maximum": np.max(sc_diagnostics.geary_c),
        "continuity weight mean": np.mean(sc_diagnostics.continuity_weight),
        "continuity weight maximum": np.max(sc_diagnostics.continuity_weight),
        "SC/feature edge overlap": sc_diagnostics.overlap_fraction,
        "automatic target attraction share": sc_diagnostics.automatic_target_share,
        "mean realized active SC share": sc_diagnostics.mean_realized_sc_share_active,
        "Phase-3 ramp iterations": sc_diagnostics.phase3_ramp_iterations,
    },
    name="SC-PaCMAP (full graph)",
)
display(diagnostic_summary.to_frame().round(4))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for axis, method in zip(axes.flat, METHOD_ORDER):
    embedding = embeddings[method]
    for class_id in class_ids:
        mask = labels_true == class_id
        axis.scatter(
            embedding[mask, 0],
            embedding[mask, 1],
            s=14,
            alpha=0.68,
            color=class_colors[class_id],
            label=EXPECTED_LABELS[class_id],
            linewidth=0,
        )
    axis.set_title(method)
    axis.set_xlabel("Embedding 1")
    axis.set_ylabel("Embedding 2")
    axis.grid(alpha=0.25)
axes.flat[-1].set_visible(False)
handles, legend_labels = axes.flat[0].get_legend_handles_labels()
fig.legend(
    handles,
    legend_labels,
    loc="lower center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 0.01),
)
fig.suptitle("Synthetic embeddings colored by evaluation-only facies", fontsize=15)
plt.tight_layout(rect=(0, 0.08, 1, 0.96))
plt.show()

## Evaluation measures

As in the CMiO notebook, K-means receives each raw 2-D embedding directly,
without further standardization, normalization, rotation, or alignment.
Spatial continuity uses each sample's 8 nearest non-self raw-XY neighbours:

$$
\frac{H}{N}=-\frac{1}{N}\sum_{i=1}^{N}\sum_j p_{ij}\log p_{ij}.
$$

Multivariate delineation uses the standardized 50-feature space:

$$
\frac{\mathrm{WCSS}}{\mathrm{TSS}}=
\frac{\sum_k\sum_{i\in k}\lVert x_i-\bar{x}_k\rVert^2}
{\sum_i\lVert x_i-\bar{x}\rVert^2}.
$$

Lower entropy and WCSS/TSS are preferable; ARI remains a held-out four-facies
reference only. The held-out-color embedding plot and original-XY partition
maps remain visible so that elongated or map-like solutions are not judged by
ARI alone.

## Result board

For each embedding and each $K=2,\ldots,10$, deterministic K-means uses 20
initializations. The board reports the spatial-continuity, multivariate-
delineation, and held-out agreement measures for the complete candidate set.

In [ ]:
# CMiO-matched evaluation: raw 2-D embeddings enter K-means unchanged.
spatial_model = NearestNeighbors(metric="euclidean", n_jobs=1).fit(
    coordinates_raw
)
spatial_neighbors = spatial_model.kneighbors(
    coordinates_raw,
    n_neighbors=N_NEIGHBORS + 1,
    return_distance=False,
)[:, 1:]
assert spatial_neighbors.shape == (len(data), N_NEIGHBORS)

spatial_context = pd.Series(
    {
        "raw-XY neighbors per sample": N_NEIGHBORS,
        "focal sample excluded": True,
        "K-means embedding input": "raw 2-D embedding (not standardized)",
    },
    name="CMiO-matched evaluation context",
)
display(spatial_context.to_frame())


def total_windowed_entropy(cluster_labels: np.ndarray) -> float:
    total = 0.0
    for neighbors in spatial_neighbors:
        counts = np.bincount(cluster_labels[neighbors])
        probabilities = counts[counts > 0] / N_NEIGHBORS
        total -= float(np.sum(probabilities * np.log(probabilities)))
    return total


def feature_space_wcss(
    cluster_labels: np.ndarray,
    n_clusters: int,
) -> float:
    centroids = np.vstack(
        [
            features_standardized[cluster_labels == cluster_id].mean(axis=0)
            for cluster_id in range(n_clusters)
        ]
    )
    residuals = features_standardized - centroids[cluster_labels]
    return float(np.sum(residuals * residuals))


result_rows = []
for method in METHOD_ORDER:
    raw_embedding = embeddings[method]
    assert raw_embedding.shape == (len(data), N_COMPONENTS)
    for k in K_VALUES:
        cluster_labels = KMeans(
            n_clusters=k,
            init="k-means++",
            n_init=20,
            random_state=1000 + k,
        ).fit_predict(raw_embedding)
        result_rows.append(
            {
                "method": method,
                "K": k,
                "H_per_sample": total_windowed_entropy(cluster_labels) / len(data),
                "WCSS_TSS": feature_space_wcss(cluster_labels, k)
                / total_sum_squares,
                "ARI": adjusted_rand_score(labels_true, cluster_labels),
            }
        )

results = pd.DataFrame(result_rows)
results["method"] = pd.Categorical(
    results["method"], categories=METHOD_ORDER, ordered=True
)
results = results.sort_values(["method", "K"]).reset_index(drop=True)

assert results.columns.tolist() == [
    "method",
    "K",
    "H_per_sample",
    "WCSS_TSS",
    "ARI",
]
assert len(results) == len(METHOD_ORDER) * len(list(K_VALUES)) == 63
assert not results.duplicated(["method", "K"]).any()
assert results.groupby("method", observed=True)["K"].apply(
    lambda values: list(values) == list(K_VALUES)
).all()
assert np.isfinite(results[["H_per_sample", "WCSS_TSS", "ARI"]]).all().all()
assert results["H_per_sample"].ge(0).all()
assert results["WCSS_TSS"].gt(0).all()
assert results["ARI"].between(-1, 1).all()

display(results.round({"H_per_sample": 4, "WCSS_TSS": 4, "ARI": 4}))

In [ ]:
known_k_results = results.loc[
    results["K"] == KNOWN_CLASS_COUNT,
    ["method", "H_per_sample", "WCSS_TSS", "ARI"],
].copy()
display(known_k_results.round({"H_per_sample": 4, "WCSS_TSS": 4, "ARI": 4}))

fingerprint_table = results.sort_values(["method", "K"]).copy()
fingerprint_table["method"] = fingerprint_table["method"].astype(str)
fingerprint_payload = fingerprint_table.to_csv(
    index=False,
    float_format="%.12g",
    lineterminator="\n",
).encode("utf-8")
metric_fingerprint = hashlib.sha256(fingerprint_payload).hexdigest()
print(f"Metric-table SHA-256: {metric_fingerprint}")

In [ ]:
# K=4 PaCMAP ablation clusters mapped back to original XY
ABLATION_METHOD_ORDER = [
    "PaCMAP feature-only",
    "PaCMAP + XY",
    "SC-PaCMAP (neighbour replacement graph)",
    "SC-PaCMAP (full graph)",
]
ablation_cluster_labels = {
    method: KMeans(
        n_clusters=KNOWN_CLASS_COUNT,
        init="k-means++",
        n_init=20,
        random_state=1000 + KNOWN_CLASS_COUNT,
    ).fit_predict(embedding_cluster_inputs[method])
    for method in ABLATION_METHOD_ORDER
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4.4), constrained_layout=True)
for axis, method in zip(axes, ABLATION_METHOD_ORDER):
    axis.scatter(
        coordinates_raw[:, 0],
        coordinates_raw[:, 1],
        c=ablation_cluster_labels[method],
        cmap="tab10",
        s=10,
        alpha=0.72,
        linewidth=0,
    )
    axis.set_title(method)
    axis.set_xlabel("X (m)")
    axis.set_ylabel("Y (m)")
    axis.ticklabel_format(style="plain", useOffset=False)
fig.suptitle(
    f"K-means partitions at K={KNOWN_CLASS_COUNT} in original XY",
    fontsize=14,
)
plt.show()

## PaCMAP-focused ablation

The focused comparison maps feature-only PaCMAP, PaCMAP + XY, and both SC-PaCMAP variants
back to original XY at $K=4$, then compares their metric trajectories and
spatial–multivariate trade-off.

In [ ]:
# PaCMAP-focused ablation across K
method_colors = dict(
    zip(METHOD_ORDER, sns.color_palette("colorblind", len(METHOD_ORDER)))
)
method_markers = dict(zip(METHOD_ORDER, ["o", "s", "^", "D", "h", "*", "X"]))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), constrained_layout=True)
metric_specs = [
    (
        "H_per_sample",
        "Mean windowed entropy, H/N",
        "Spatial continuity (lower is better)",
    ),
    (
        "WCSS_TSS",
        "WCSS / TSS",
        "Multivariate delineation (lower is better)",
    ),
    (
        "ARI",
        "Adjusted Rand index",
        "Held-out facies agreement (higher is better)",
    ),
]
for axis, (metric, ylabel, title) in zip(axes, metric_specs):
    for method in ABLATION_METHOD_ORDER:
        method_results = results.loc[results["method"] == method].sort_values("K")
        axis.plot(
            method_results["K"],
            method_results[metric],
            color=method_colors[method],
            marker=method_markers[method],
            linewidth=2,
            markersize=6,
            label=method,
        )
    axis.axvline(
        KNOWN_CLASS_COUNT,
        color="black",
        linewidth=1,
        linestyle="--",
        alpha=0.6,
    )
    if metric == "ARI":
        axis.axhline(0, color="black", linewidth=1, linestyle=":")
    axis.set(xlabel="K-means clusters (K)", ylabel=ylabel, title=title)
    axis.set_xticks(list(K_VALUES))
    axis.grid(alpha=0.25)
handles, legend_labels = axes[0].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc="outside lower center", ncol=3, frameon=False)
fig.suptitle("PaCMAP-focused ablation", fontsize=15)
plt.show()

### PaCMAP ablation trade-off

The highlighted ring marks $K=4$. Movement toward the lower-left combines more
contiguous spatial partitions with tighter multivariate clusters; the paths
show how this balance changes with K.

In [ ]:
marker_sizes = {k: 45 + 22 * (k - min(K_VALUES)) for k in K_VALUES}

_, axis = plt.subplots(figsize=(9, 6.3))
for method in ABLATION_METHOD_ORDER:
    method_results = results.loc[results["method"] == method].sort_values("K")
    axis.plot(
        method_results["H_per_sample"],
        method_results["WCSS_TSS"],
        color=method_colors[method],
        linewidth=1.5,
        alpha=0.6,
    )
    axis.scatter(
        method_results["H_per_sample"],
        method_results["WCSS_TSS"],
        s=[marker_sizes[k] for k in method_results["K"]],
        marker=method_markers[method],
        color=method_colors[method],
        edgecolor="white",
        linewidth=0.7,
        alpha=0.9,
        label=method,
        zorder=3,
    )
    known_point = method_results.loc[
        method_results["K"] == KNOWN_CLASS_COUNT
    ]
    axis.scatter(
        known_point["H_per_sample"],
        known_point["WCSS_TSS"],
        s=marker_sizes[KNOWN_CLASS_COUNT] + 35,
        marker=method_markers[method],
        facecolor="none",
        edgecolor="black",
        linewidth=1.8,
        zorder=4,
    )

axis.set(
    xlabel="Mean windowed entropy H/N (lower is better)",
    ylabel="WCSS/TSS in standardized feature space (lower is better)",
    title="PaCMAP-focused spatial–multivariate trade-off",
)
axis.grid(alpha=0.3)
axis.legend(title="Method", frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

## Full seven-method benchmark

This section extends the same deterministic setup and evaluation to UMAP + XY,
LocalMAP + XY, and SpaSNE. The input pathways remain explicit: the three
`+ XY` candidates use standardized coordinates as ordinary input columns,
feature-only PaCMAP receives no coordinates, the replacement-graph and full-graph
SC-PaCMAP variants use distinct spatial graph structures, and SpaSNE receives raw XY
separately through its spatial objective.

In [ ]:
# All candidates across K
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), constrained_layout=True)
metric_specs = [
    (
        "H_per_sample",
        "Mean windowed entropy, H/N",
        "Spatial continuity (lower is better)",
    ),
    (
        "WCSS_TSS",
        "WCSS / TSS",
        "Multivariate delineation (lower is better)",
    ),
    (
        "ARI",
        "Adjusted Rand index",
        "Held-out facies agreement (higher is better)",
    ),
]
for axis, (metric, ylabel, title) in zip(axes, metric_specs):
    for method in METHOD_ORDER:
        method_results = results.loc[results["method"] == method].sort_values("K")
        axis.plot(
            method_results["K"],
            method_results[metric],
            color=method_colors[method],
            marker=method_markers[method],
            linewidth=2,
            markersize=5.5,
            label=method,
        )
    axis.axvline(
        KNOWN_CLASS_COUNT,
        color="black",
        linewidth=1,
        linestyle="--",
        alpha=0.6,
    )
    if metric == "ARI":
        axis.axhline(0, color="black", linewidth=1, linestyle=":")
    axis.set(xlabel="K-means clusters (K)", ylabel=ylabel, title=title)
    axis.set_xticks(list(K_VALUES))
    axis.grid(alpha=0.25)
handles, legend_labels = axes[0].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc="outside lower center", ncol=3, frameon=False)
fig.suptitle("Full seven-method benchmark", fontsize=15)
plt.show()

### Full benchmark trade-off

Each point is a method-by-$K$ solution; marker size increases with $K$, and the
black ring identifies the four-class reference. The lower-left direction is
preferable jointly, but no single scalar collapses the two evaluation
objectives.

In [ ]:
# All-candidate spatial-multivariate trade-off
_, axis = plt.subplots(figsize=(10.5, 6.8))
for method in METHOD_ORDER:
    method_results = results.loc[results["method"] == method].sort_values("K")
    axis.plot(
        method_results["H_per_sample"],
        method_results["WCSS_TSS"],
        color=method_colors[method],
        linewidth=1.4,
        alpha=0.55,
    )
    axis.scatter(
        method_results["H_per_sample"],
        method_results["WCSS_TSS"],
        s=[marker_sizes[k] for k in method_results["K"]],
        marker=method_markers[method],
        color=method_colors[method],
        edgecolor="white",
        linewidth=0.7,
        alpha=0.9,
        label=method,
        zorder=3,
    )
    known_point = method_results.loc[
        method_results["K"] == KNOWN_CLASS_COUNT
    ]
    axis.scatter(
        known_point["H_per_sample"],
        known_point["WCSS_TSS"],
        s=marker_sizes[KNOWN_CLASS_COUNT] + 35,
        marker=method_markers[method],
        facecolor="none",
        edgecolor="black",
        linewidth=1.8,
        zorder=4,
    )
axis.set(
    xlabel="Mean windowed entropy H/N (lower is better)",
    ylabel="WCSS/TSS in standardized feature space (lower is better)",
    title="Full seven-method spatial–multivariate trade-off",
)
axis.annotate(
    "Jointly lower",
    xy=(0.035, 0.04),
    xytext=(0.18, 0.17),
    xycoords="axes fraction",
    arrowprops={"arrowstyle": "->", "color": "black"},
)
axis.grid(alpha=0.3)
axis.legend(
    title="Method",
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.01, 1.0),
)
plt.tight_layout(rect=(0, 0, 0.8, 1))
plt.show()

## Summary at the four-class reference

The summary below identifies the best observed value for each metric at $K=4$
and quantifies the SC-PaCMAP (full graph) change relative to feature-only PaCMAP. These
statements describe the known reference count; the held-out labels did not
select an embedding, tune a method, or choose K within the reported sweep.
Spatial entropy, multivariate WCSS/TSS, ARI, the truth-colored embeddings, and
the original-XY maps should be interpreted together.

In [ ]:
best_entropy = known_k_results.loc[known_k_results["H_per_sample"].idxmin()]
best_wcss = known_k_results.loc[known_k_results["WCSS_TSS"].idxmin()]
best_ari = known_k_results.loc[known_k_results["ARI"].idxmax()]
overall_ari_best = results.loc[results["ARI"].idxmax()]
sc_known = known_k_results.loc[
    known_k_results["method"] == "SC-PaCMAP (full graph)"
].iloc[0]
feature_known = known_k_results.loc[
    known_k_results["method"] == "PaCMAP feature-only"
].iloc[0]
entropy_reduction = 100 * (
    feature_known["H_per_sample"] - sc_known["H_per_sample"]
) / feature_known["H_per_sample"]
wcss_change = 100 * (
    sc_known["WCSS_TSS"] - feature_known["WCSS_TSS"]
) / feature_known["WCSS_TSS"]

print(
    f"At K={KNOWN_CLASS_COUNT}, {best_entropy['method']} has the lowest entropy "
    f"H/N ({best_entropy['H_per_sample']:.4f}), {best_wcss['method']} has the "
    f"lowest WCSS/TSS ({best_wcss['WCSS_TSS']:.4f}), and {best_ari['method']} "
    f"has the highest ARI ({best_ari['ARI']:.4f})."
)
print(
    f"Relative to feature-only PaCMAP at K={KNOWN_CLASS_COUNT}, SC-PaCMAP (full graph) "
    f"reduces entropy by {entropy_reduction:.1f}% while changing WCSS by "
    f"{wcss_change:+.1f}%."
)
print(
    f"The largest ARI in the complete sweep is {overall_ari_best['ARI']:.4f} "
    f"from {overall_ari_best['method']} at K={int(overall_ari_best['K'])}."
)

In [ ]:
####################### PUBLICATION NEEDED STATISTICS #########################################

## Confirmatory replicated synthetic experiment

Ten **data-generation seeds** create ten independently sampled datasets from the fixed repeated-facies generator. Within each dataset, three **algorithm seeds** quantify stochastic embedding variability for all seven benchmark methods.

The prespecified primary comparison remains SC-PaCMAP (full graph) versus feature-only PaCMAP at the known synthetic reference $K=4$. The replacement-graph variant is included as a descriptive structural ablation. Spatial entropy is the primary endpoint; WCSS/TSS and ARI describe the associated multivariate and truth-agreement trade-offs.

Inference uses the ten $K=4$ realization-level medians, not the nested algorithm runs. The additional $K=2,\ldots,10$ results support descriptive trajectory and trade-off analysis only. Paired differences at $K=4$ are reported with a bootstrap 95% confidence interval and an exact two-sided paired sign-flip permutation test.

In [ ]:
DATA_SEEDS = tuple(range(10))
ALGORITHM_SEEDS = (11, 42, 97)
REPLICATED_K = KNOWN_CLASS_COUNT
REPLICATED_K_VALUES = tuple(K_VALUES)
REPLICATED_METHOD_ORDER = list(METHOD_ORDER)
REPLICATED_RESULTS_PATH = ROOT / "results" / "synthetic_replicated_runs.csv"

settings = PaCMAPSettings(
    n_components=N_COMPONENTS,
    n_neighbors=N_NEIGHBORS,
    mn_ratio=0.5,
    fp_ratio=2.0,
    learning_rate=1.0,
    num_iters=(100, 100, 250),
    random_state=SEED,
    knn_backend="faiss",
    verbose=False,
 )
sc_config = SCPaCMAPConfig()
pacmap_parameters = {
    "n_components": N_COMPONENTS,
    "n_neighbors": N_NEIGHBORS,
    "MN_ratio": 0.5,
    "FP_ratio": 2.0,
    "distance": "euclidean",
    "lr": 1.0,
    "num_iters": (100, 100, 250),
    "apply_pca": True,
    "save_tree": False,
    "knn_backend": "faiss",
}

print(
    f"Confirmatory design: {len(DATA_SEEDS)} generated datasets x "
    f"{len(ALGORITHM_SEEDS)} algorithm seeds; K={min(REPLICATED_K_VALUES)}-"
    f"{max(REPLICATED_K_VALUES)}, confirmatory K={REPLICATED_K}."
 )

In [ ]:
from dataclasses import replace
from itertools import product

DATA_SEEDS = tuple(range(10))
ALGORITHM_SEEDS = (11, 42, 97)
REPLICATED_K = KNOWN_CLASS_COUNT
REPLICATED_K_VALUES = tuple(K_VALUES)
REPLICATED_METHOD_ORDER = list(METHOD_ORDER)
REPLICATED_RESULTS_PATH = ROOT / "results" / "synthetic_replicated_runs.csv"


def evaluate_replicated_partition(
    cluster_labels: np.ndarray,
    feature_values: np.ndarray,
    spatial_neighbor_indices: np.ndarray,
    truth: np.ndarray,
    n_clusters: int,
    tss: float,
 ) -> dict[str, float]:
    entropy_total = 0.0
    neighborhood_size = spatial_neighbor_indices.shape[1]
    for neighbors in spatial_neighbor_indices:
        counts = np.bincount(cluster_labels[neighbors], minlength=n_clusters)
        probabilities = counts[counts > 0] / neighborhood_size
        entropy_total -= float(np.sum(probabilities * np.log(probabilities)))

    centroids = np.vstack(
        [
            feature_values[cluster_labels == cluster_id].mean(axis=0)
            for cluster_id in range(n_clusters)
        ]
    )
    residuals = feature_values - centroids[cluster_labels]
    return {
        "H_per_sample": entropy_total / len(cluster_labels),
        "WCSS_TSS": float(np.sum(residuals * residuals)) / tss,
        "ARI": adjusted_rand_score(truth, cluster_labels),
    }


def fit_replicated_localmap(
    x_input: np.ndarray,
    n_neighbors: int,
    random_state: int,
 ) -> np.ndarray:
    import subprocess

    worker_code = r'''
import os
os.environ["NUMBA_NUM_THREADS"] = "1"
import importlib
import sys
import numba
import numpy as np
import pacmap

@numba.njit
def seed_numba_rng(random_state):
    np.random.seed(random_state)

pacmap_core = importlib.import_module("pacmap.pacmap")
original_resample = pacmap_core.sample_FP_pair_nearby
resample_call_index = 0

def deterministic_resample(X, pair_neighbors, old_pair_FP, Y, low_dist_thres):
    global resample_call_index
    seed_numba_rng(random_state + resample_call_index)
    resample_call_index += 1
    return original_resample(X, pair_neighbors, old_pair_FP, Y, low_dist_thres)

rows, columns, n_neighbors, random_state = map(int, sys.argv[1:5])
pacmap_core.sample_FP_pair_nearby = deterministic_resample
x_input = np.frombuffer(sys.stdin.buffer.read(), dtype=np.float64).reshape(rows, columns)
seed_numba_rng(random_state)
model = pacmap.LocalMAP(
    n_components=2, n_neighbors=n_neighbors, MN_ratio=0.5, FP_ratio=2.0,
    distance="euclidean", lr=1.0, num_iters=(100, 100, 250),
    apply_pca=True, random_state=random_state, save_tree=False, low_dist_thres=10,
    knn_backend="faiss",
)
embedding = model.fit_transform(x_input, init="pca", save_pairs=False)
sys.stdout.buffer.write(np.asarray(embedding, dtype=np.float32).tobytes())
'''
    x_input = np.ascontiguousarray(x_input, dtype=np.float64)
    completed = subprocess.run(
        [
            sys.executable,
            "-c",
            worker_code,
            *map(str, x_input.shape),
            str(n_neighbors),
            str(random_state),
        ],
        input=x_input.tobytes(),
        capture_output=True,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.decode("utf-8", errors="replace"))
    expected_bytes = len(x_input) * N_COMPONENTS * np.dtype(np.float32).itemsize
    if len(completed.stdout) != expected_bytes:
        raise RuntimeError("LocalMAP worker returned an unexpected embedding size.")
    return np.frombuffer(completed.stdout, dtype=np.float32).reshape(
        len(x_input), N_COMPONENTS
    ).copy()


replicated_rows = []
for data_seed in DATA_SEEDS:
    generated_data, _ = make_synthetic_repeated_facies(random_state=data_seed)
    generated_features_raw = generated_data[FEATURE_COLUMNS].to_numpy(dtype=np.float64)
    generated_coordinates = generated_data[XY_COLUMNS].to_numpy(dtype=np.float64)
    generated_truth = generated_data["facies_id"].to_numpy(dtype=np.int64)

    generated_features = StandardScaler().fit_transform(
        generated_features_raw
    ).astype(np.float32)
    generated_coordinates_scaled = StandardScaler().fit_transform(
        generated_coordinates
    )
    generated_features_with_xy = np.column_stack(
        [generated_features, generated_coordinates_scaled]
    )
    generated_feature_center = generated_features.mean(axis=0, keepdims=True)
    generated_tss = float(
        np.sum((generated_features - generated_feature_center) ** 2)
    )
    generated_spatial_neighbors = NearestNeighbors(
        metric="euclidean", n_jobs=1
    ).fit(generated_coordinates).kneighbors(
        generated_coordinates,
        n_neighbors=N_NEIGHBORS + 1,
        return_distance=False,
    )[:, 1:]

    for algorithm_seed in ALGORITHM_SEEDS:
        replicated_settings = replace(settings, random_state=algorithm_seed)
        replicated_pacmap_parameters = {
            **pacmap_parameters,
            "random_state": algorithm_seed,
        }
        replicated_embeddings = {}
        replicated_runtimes = {}

        started_at = time.perf_counter()
        replicated_embeddings["UMAP + XY"] = umap.UMAP(
            n_neighbors=N_NEIGHBORS,
            n_components=N_COMPONENTS,
            metric="euclidean",
            learning_rate=1.0,
            random_state=algorithm_seed,
            n_jobs=1,
        ).fit_transform(generated_features_with_xy)
        replicated_runtimes["UMAP + XY"] = time.perf_counter() - started_at

        started_at = time.perf_counter()
        replicated_embeddings["PaCMAP + XY"] = pacmap.PaCMAP(
            **replicated_pacmap_parameters
        ).fit_transform(generated_features_with_xy, init="pca", save_pairs=False)
        replicated_runtimes["PaCMAP + XY"] = time.perf_counter() - started_at

        started_at = time.perf_counter()
        replicated_embeddings["LocalMAP + XY"] = fit_replicated_localmap(
            generated_features_with_xy, N_NEIGHBORS, algorithm_seed
        )
        replicated_runtimes["LocalMAP + XY"] = time.perf_counter() - started_at

        started_at = time.perf_counter()
        replicated_embeddings["PaCMAP feature-only"] = pacmap.PaCMAP(
            **replicated_pacmap_parameters
        ).fit_transform(generated_features, init="pca", save_pairs=False)
        replicated_runtimes["PaCMAP feature-only"] = time.perf_counter() - started_at

        started_at = time.perf_counter()
        replicated_replacement_result, _ = run_legacy_sc_pacmap(
            generated_features,
            generated_coordinates,
            replicated_settings,
        )
        replicated_embeddings["SC-PaCMAP (neighbour replacement graph)"] = (
            replicated_replacement_result.embedding
        )
        replicated_runtimes["SC-PaCMAP (neighbour replacement graph)"] = (
            time.perf_counter() - started_at
        )

        started_at = time.perf_counter()
        replicated_sc_result, _ = run_sc_pacmap(
            generated_features,
            generated_coordinates,
            replicated_settings,
            config=sc_config,
        )
        replicated_embeddings["SC-PaCMAP (full graph)"] = replicated_sc_result.embedding
        replicated_runtimes["SC-PaCMAP (full graph)"] = time.perf_counter() - started_at

        started_at = time.perf_counter()
        replicated_embeddings["SpaSNE"] = python_spasne(
            generated_features,
            generated_coordinates,
            random_state=algorithm_seed,
            alpha=SPASNE_ALPHA,
            beta=SPASNE_BETA,
            exaggeration=SPASNE_EXAGGERATION,
            perplexity=SPASNE_PERPLEXITY,
            theta=SPASNE_THETA,
            max_iter=SPASNE_MAX_ITER,
            use_pca=True,
            spasne_dir=SPASNE_DIR,
        )
        replicated_runtimes["SpaSNE"] = time.perf_counter() - started_at

        assert set(replicated_embeddings) == set(METHOD_ORDER)
        for method in METHOD_ORDER:
            for k in REPLICATED_K_VALUES:
                method_labels = KMeans(
                    n_clusters=k,
                    init="k-means++",
                    n_init=20,
                    random_state=20_000 + data_seed,
                ).fit_predict(replicated_embeddings[method])
                replicated_rows.append(
                    {
                        "data_seed": data_seed,
                        "algorithm_seed": algorithm_seed,
                        "method": method,
                        "K": k,
                        "runtime_seconds": replicated_runtimes[method],
                        **evaluate_replicated_partition(
                            method_labels,
                            generated_features,
                            generated_spatial_neighbors,
                            generated_truth,
                            k,
                            generated_tss,
                        ),
                    }
                )

replicated_results = pd.DataFrame(replicated_rows)
replicated_results["method"] = pd.Categorical(
    replicated_results["method"],
    categories=REPLICATED_METHOD_ORDER,
    ordered=True,
 )
expected_rows = (
    len(DATA_SEEDS) * len(REPLICATED_K_VALUES)
    * len(METHOD_ORDER) * len(ALGORITHM_SEEDS)
 )
assert len(replicated_results) == expected_rows == 1_890
assert not replicated_results.duplicated(
    ["data_seed", "algorithm_seed", "method", "K"]
 ).any()
assert np.isfinite(
    replicated_results[["H_per_sample", "WCSS_TSS", "ARI"]]
 ).all().all()

REPLICATED_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
replicated_results.to_csv(REPLICATED_RESULTS_PATH, index=False)
display(replicated_results.groupby("method", observed=True).size().rename("rows").to_frame())
print(f"Saved {len(replicated_results)} run-level rows to {REPLICATED_RESULTS_PATH}")

In [ ]:
METRICS = ("H_per_sample", "WCSS_TSS", "ARI")
PRIMARY_METHOD = "SC-PaCMAP (full graph)"
PRIMARY_COMPARATOR = "PaCMAP feature-only"

realization_results = (
    replicated_results
    .groupby(["data_seed", "method", "K"], observed=True)[list(METRICS)]
    .median()
    .reset_index()
 )
assert realization_results.groupby(["method", "K"], observed=True)["data_seed"].nunique().eq(
    len(DATA_SEEDS)
 ).all()
reference_realization_results = realization_results.loc[
    realization_results["K"] == REPLICATED_K
].copy()

realization_summary = (
    reference_realization_results
    .groupby("method", observed=True)[list(METRICS)]
    .agg(["median", lambda values: values.quantile(0.25), lambda values: values.quantile(0.75)])
 )
realization_summary.columns = [
    f"{metric}_{statistic}"
    for metric, statistic in realization_summary.columns
 ]
realization_summary = realization_summary.rename(
    columns=lambda column: column.replace("<lambda_0>", "q25").replace("<lambda_1>", "q75")
 )


def exact_sign_flip_pvalue(differences: np.ndarray) -> float:
    differences = np.asarray(differences, dtype=np.float64)
    observed = abs(float(np.mean(differences)))
    permuted = np.array(
        [
            abs(float(np.mean(differences * np.asarray(signs))))
            for signs in product((-1.0, 1.0), repeat=len(differences))
        ]
    )
    return float(np.mean(permuted >= observed - 1e-15))


def paired_bootstrap_median_ci(
    differences: np.ndarray,
    random_state: int = 2026,
    repetitions: int = 20_000,
 ) -> tuple[float, float]:
    differences = np.asarray(differences, dtype=np.float64)
    rng = np.random.default_rng(random_state)
    indices = rng.integers(0, len(differences), size=(repetitions, len(differences)))
    bootstrap_medians = np.median(differences[indices], axis=1)
    lower, upper = np.quantile(bootstrap_medians, [0.025, 0.975])
    return float(lower), float(upper)


replicated_wide = reference_realization_results.pivot(
    index="data_seed", columns="method", values=list(METRICS)
 )
contrast_rows = []
for metric_index, metric in enumerate(METRICS):
    differences = (
        replicated_wide[(metric, PRIMARY_METHOD)]
        - replicated_wide[(metric, PRIMARY_COMPARATOR)]
    ).to_numpy()
    ci_low, ci_high = paired_bootstrap_median_ci(
        differences, random_state=2026 + metric_index
    )
    contrast_rows.append(
        {
            "metric": metric,
            "contrast": f"{PRIMARY_METHOD} - {PRIMARY_COMPARATOR}",
            "median_difference": float(np.median(differences)),
            "bootstrap_95_ci_low": ci_low,
            "bootstrap_95_ci_high": ci_high,
            "mean_difference": float(np.mean(differences)),
            "exact_two_sided_p": exact_sign_flip_pvalue(differences),
            "SC_wins": int(
                np.sum(differences < 0) if metric != "ARI" else np.sum(differences > 0)
            ),
            "realizations": len(differences),
        }
    )

contrast_results = pd.DataFrame(contrast_rows)
ordered_p = contrast_results["exact_two_sided_p"].sort_values().index
holm_adjusted = pd.Series(index=contrast_results.index, dtype=np.float64)
running_maximum = 0.0
for rank, row_index in enumerate(ordered_p):
    adjusted = min(
        1.0,
        contrast_results.loc[row_index, "exact_two_sided_p"]
        * (len(contrast_results) - rank),
    )
    running_maximum = max(running_maximum, adjusted)
    holm_adjusted.loc[row_index] = running_maximum
contrast_results["holm_adjusted_p"] = holm_adjusted

display(realization_summary.round(4))
display(contrast_results.round(4))

## All-K stability bands

Each dataset realization (median across the three algorithm seeds) is evaluated across $K=2,\ldots,10$. The solid line is the across-dataset median and the shaded band is the min-max range at each $K$. These bands describe dataset-to-dataset and optimizer sensitivity across the ten generated datasets; they are **not confidence intervals**, and formal inference remains restricted to the prespecified $K=4$ comparison reported above.

In [ ]:
replicated_results = pd.read_csv(ROOT / "results" / "synthetic_replicated_runs.csv")
realization_by_k = (
    replicated_results
    .groupby(["data_seed", "method", "K"])[["H_per_sample", "WCSS_TSS", "ARI"]]
    .median()
    .reset_index()
)
method_styles = {
    "UMAP + XY": ("#D55E00", "o", "-"),
    "PaCMAP + XY": ("#56B4E9", "^", "-"),
    "LocalMAP + XY": ("#009E73", "v", "-"),
    "PaCMAP feature-only": ("#7A5195", "P", "-"),
    "SC-PaCMAP (neighbour replacement graph)": ("#E15759", "h", "-"),
    "SC-PaCMAP (full graph)": ("#E60049", "*", "-"),
    "SpaSNE": ("#D4A900", "X", "-"),
}
metrics = [
    ("H_per_sample", "Mean windowed entropy H/N", "lower is better"),
    ("WCSS_TSS", "WCSS/TSS", "lower is better"),
    ("ARI", "Adjusted Rand index", "higher is better"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.7), constrained_layout=True)
for axis, (metric, ylabel, direction) in zip(axes, metrics):
    for method in REPLICATED_METHOD_ORDER:
        method_rows = realization_by_k.loc[realization_by_k["method"] == method]
        color, marker, linestyle = method_styles[method]
        is_sc = method == "SC-PaCMAP (full graph)"
        summary = method_rows.groupby("K")[metric].agg(["median", "min", "max"])

        axis.fill_between(
            summary.index, summary["min"], summary["max"],
            color=color, alpha=0.16 if is_sc else 0.07, linewidth=0,
        )
        axis.plot(
            summary.index, summary["median"], color=color, marker=marker,
            linestyle=linestyle, linewidth=2.8 if is_sc else 1.5,
            markersize=8 if is_sc else 5, label=method,
        )

    axis.axvline(KNOWN_CLASS_COUNT, color="#333333", linestyle=":", linewidth=1.2)
    axis.set(
        xlabel="Number of clusters K", ylabel=f"{ylabel} ({direction})",
        xticks=list(REPLICATED_K_VALUES),
    )
    axis.grid(alpha=0.2)

fig.suptitle("Synthetic all-K stability across ten generated datasets", fontsize=14)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels, loc="outside lower center", ncol=3,
    title="Lines: dataset medians; bands: min-max across 10 datasets",
)
plt.show()

### Reference-K stability cloud

Each faint point is one (data-seed, algorithm-seed) run at the reference $K=4$; the larger outlined marker is the across-dataset median. The cloud shows how much data-generation and optimizer randomness jointly move the spatial-feature trade-off at the reference $K$.

In [ ]:
method_styles = {  # method: (color, marker)
    "UMAP + XY": ("#D55E00", "o"),
    "PaCMAP + XY": ("#56B4E9", "^"),
    "LocalMAP + XY": ("#009E73", "v"),
    "PaCMAP feature-only": ("#7A5195", "P"),
    "SC-PaCMAP (neighbour replacement graph)": ("#E15759", "h"),
    "SC-PaCMAP (full graph)": ("#E60049", "*"),
    "SpaSNE": ("#D4A900", "X"),
}

# Reload saved results so the plot works after a kernel restart.
replicated_results = pd.read_csv(
    ROOT / "results" / "synthetic_replicated_runs.csv"
 )
reference_runs = replicated_results.loc[
    replicated_results["K"] == KNOWN_CLASS_COUNT
].copy()
realization_results = (
    reference_runs
    .groupby(["data_seed", "method"])[["H_per_sample", "WCSS_TSS", "ARI"]]
    .median()
    .reset_index()
 )

_, axis = plt.subplots(figsize=(10.5, 7.0))


def draw_points(x, y, method, size, alpha, edge, line_width, z_order, label=None):
    color, marker = method_styles[method]
    axis.scatter(
        x, y,
        s=size, marker=marker, color=color, alpha=alpha,
        edgecolor=edge, linewidth=line_width, zorder=z_order, label=label,
    )


for method in method_styles:
    runs = reference_runs[reference_runs["method"] == method]
    datasets = realization_results[realization_results["method"] == method]
    is_sc = method == "SC-PaCMAP (full graph)"

    # Individual runs, independent-dataset medians, and the overall median.
    draw_points(
        runs["H_per_sample"], runs["WCSS_TSS"], method,
        58 if is_sc else 26, 0.62 if is_sc else 0.28,
        "none", 0, 2,
    )
    draw_points(
        datasets["H_per_sample"], datasets["WCSS_TSS"], method,
        150 if is_sc else 68, 1.0 if is_sc else 0.9,
        "black", 1.1 if is_sc else 0.7, 6 if is_sc else 3, method,
    )
    draw_points(
        datasets["H_per_sample"].median(), datasets["WCSS_TSS"].median(), method,
        360 if is_sc else 185, 1.0,
        "black", 1.8 if is_sc else 1.35, 8 if is_sc else 5,
    )

sc_data = realization_results[realization_results["method"] == "SC-PaCMAP (full graph)"]
sc_median = (sc_data["H_per_sample"].median(), sc_data["WCSS_TSS"].median())
sc_color = method_styles["SC-PaCMAP (full graph)"][0]
axis.annotate(
    "SC-PaCMAP (full graph)", xy=sc_median, xytext=(34, -34), textcoords="offset points",
    color=sc_color, fontsize=11, fontweight="bold",
    arrowprops={"arrowstyle": "->", "color": sc_color, "lw": 1.8},
    zorder=9,
 )

axis.set(
    xlabel="Mean windowed entropy H/N (lower is better)",
    ylabel="WCSS/TSS in standardized feature space (lower is better)",
    title=(
        "Replicated synthetic spatial-feature trade-off at K=4\n"
        "Faint points: runs; outlined points: dataset medians; "
        "large symbols: overall medians"
    ),
 )
axis.annotate(
    "Jointly lower", xy=(0.035, 0.045), xytext=(0.18, 0.17),
    xycoords="axes fraction",
    arrowprops={"arrowstyle": "->", "color": "black"},
 )
axis.grid(alpha=0.25)
axis.legend(
    frameon=False, fontsize=8.5, loc="upper center",
    bbox_to_anchor=(0.5, -0.13), ncol=4,
 )
plt.tight_layout()
plt.show()

### All-K trade-off range

Marker size increases with $K$. Each faint line is one data-seed's path across K, and the bold line with markers traces the across-dataset median. This keeps the joint spatial-feature trend readable instead of drawing many equally-dark dataset paths.

In [ ]:
# Reload all-K results so this figure works after a kernel restart.
replicated_results = pd.read_csv(
    ROOT / "results" / "synthetic_replicated_runs.csv"
)
all_k_realizations = (
    replicated_results
    .groupby(["data_seed", "method", "K"])[["H_per_sample", "WCSS_TSS", "ARI"]]
    .median()
    .reset_index()
)
method_styles = {
    "UMAP + XY": ("#D55E00", "o", "-"),
    "PaCMAP + XY": ("#56B4E9", "^", "-"),
    "LocalMAP + XY": ("#009E73", "v", "-"),
    "PaCMAP feature-only": ("#7A5195", "P", "-"),
    "SC-PaCMAP (neighbour replacement graph)": ("#E15759", "h", "-"),
    "SC-PaCMAP (full graph)": ("#E60049", "*", "-"),
    "SpaSNE": ("#D4A900", "X", "-"),
}

fig, axis = plt.subplots(figsize=(11.5, 7.5), constrained_layout=True)
for method, (color, marker, linestyle) in method_styles.items():
    method_rows = all_k_realizations.loc[all_k_realizations["method"] == method]
    is_sc = method == "SC-PaCMAP (full graph)"

    # One faint colored line per data seed, instead of a solid shaded box per K.
    for seed in DATA_SEEDS:
        seed_rows = method_rows.loc[method_rows["data_seed"] == seed].sort_values("K")
        axis.plot(
            seed_rows["H_per_sample"], seed_rows["WCSS_TSS"], color=color,
            linewidth=1.0, alpha=0.28, zorder=1,
        )

    summary = method_rows.groupby("K")[["H_per_sample", "WCSS_TSS"]].median()
    sizes = 28 + 7 * (summary.index - summary.index.min())
    axis.plot(
        summary["H_per_sample"], summary["WCSS_TSS"], color=color,
        linestyle=linestyle, linewidth=3.0 if is_sc else 1.7,
        zorder=2, label=method,
    )
    axis.scatter(
        summary["H_per_sample"], summary["WCSS_TSS"], s=sizes,
        color=color, marker=marker,
        edgecolor="black" if is_sc else color,
        linewidth=0.9 if is_sc else 0, zorder=3,
    )

    if is_sc:
        for k in (int(summary.index.min()), KNOWN_CLASS_COUNT, int(summary.index.max())):
            axis.annotate(
                f"K={k}", (summary.loc[k, "H_per_sample"], summary.loc[k, "WCSS_TSS"]),
                xytext=(6, 6), textcoords="offset points", color=color,
                fontsize=9, fontweight="bold",
            )

axis.set(
    xlabel="Mean windowed entropy H/N (lower is better)",
    ylabel="WCSS/TSS in standardized feature space (lower is better)",
    title="Synthetic all-K spatial-feature trade-off range",
)
axis.grid(alpha=0.25)
axis.legend(
    title="Faint lines: dataset paths; bold line: dataset median",
    frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left",
)
plt.show()

### Confirmatory interpretation

Across the ten independently generated datasets, SC-PaCMAP (full graph) improves on feature-only PaCMAP in all ten realization-level comparisons: the median paired differences are $-0.2409$ for $H/N$, $-0.0246$ for WCSS/TSS, and $+0.3098$ for ARI. The bootstrap 95% intervals exclude zero, and each exact paired sign-flip test remains significant after Holm correction ($p_{\mathrm{Holm}}=0.0059$). The cloud additionally shows all seven benchmark embeddings, including the replacement-graph structural ablation, as descriptive context; no pairwise significance claims are made for those exploratory comparisons.

At $K=4$, the neighbour replacement graph has the strongest descriptive median across the three reported endpoints. Formal inference remains limited to the prespecified full-graph comparison with feature-only PaCMAP; the replacement-graph result is descriptive. Therefore, this experiment supports the narrower claim that full-graph spatial-context injection improves the tested **two-dimensional PaCMAP representation** under this repeated-facies generator. It does not establish universal superiority outside this fixed data-generating process.

The all-$K$ trajectories provide a robustness view of how each spatial-feature compromise changes with cluster count. They do not replace the $K=4$ cloud: the cloud matches the prespecified scientific contrast and supports the reported paired inference, whereas the trajectories are exploratory context.